# ResNet-18 on CIFAR-10
Reproduction of He et al. (2015) on CIFAR-10.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
PROJECT_DIR = '/content/drive/MyDrive/ResNet_Cifar10'
os.chdir(PROJECT_DIR)
!pwd

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Not available')
print('CUDA:', torch.version.cuda)

In [ ]:
# First run — no resume
!cd {PROJECT_DIR} && python src/train.py --config configs/baseline.yaml

In [ ]:
# Resume after a disconnect — update the timestamp path
# !cd {PROJECT_DIR} && python src/train.py --config configs/baseline.yaml \
#     --resume results/baseline/<timestamp>/model_last.pth

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import glob

# Pick the most recent run automatically
csv_files = sorted(glob.glob('results/baseline/*/metrics.csv'))
assert csv_files, 'No metrics.csv found — run training first'
metrics_path = csv_files[-1]
print('Reading:', metrics_path)

df = pd.read_csv(metrics_path)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(df['epoch'], df['train_loss'], label='Train')
ax1.plot(df['epoch'], df['test_loss'],  label='Test')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Loss')
ax1.legend()

ax2.plot(df['epoch'], df['train_acc'], label='Train')
ax2.plot(df['epoch'], df['test_acc'],  label='Test')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Accuracy')
ax2.legend()

plt.tight_layout()
plt.savefig('results/baseline/curves.png', dpi=150)
plt.show()
print(f"Best test acc: {df['test_acc'].max():.4f} at epoch {df['test_acc'].idxmax()+1}")

In [ ]:
# Evaluate best checkpoint
best_ckpt = sorted(glob.glob('results/baseline/*/model_best.pth'))[-1]
!cd {PROJECT_DIR} && python src/evaluate.py --config configs/baseline.yaml --checkpoint {best_ckpt}